# Sprint 4 Experiment — [MODEL] × [DATASET] × [PROMPT]

**Fill in the three CONFIG cells below, then Run All.**

| Setting | Value |
|---------|-------|
| Model | ← set in Cell 2 |
| Dataset | ← set in Cell 2 |
| Prompt | ← set in Cell 2 |
| Chunk size | 3000 (author default — fixed) |
| Chunk overlap | 300 (author default — fixed) |
| Top-K | 5 (author default — fixed) |
| Temperature | 0.1 (author default — fixed) |
| Embedding | all-MiniLM-L6-v2 (local, free) |

## Cell 1 — Setup paths

In [ ]:
import sys, os

# ── Absolute paths (hardcoded for reliability) ───────────────────────────────
project_root = '/Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026'
sprint4_root = os.path.join(project_root, 'Sprint 4')
sprint3_uda  = os.path.join(project_root, 'Sprint 3', 'UDA-Benchmark')

for p in [sprint4_root, sprint3_uda]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(sprint3_uda)  # UDA preprocess uses relative paths from this root
print(f'project_root : {project_root}')
print(f'sprint4_root : {sprint4_root}')
print(f'sprint3_uda  : {sprint3_uda}')
print(f'cwd          : {os.getcwd()}')

if not os.path.isdir(sprint3_uda):
    raise FileNotFoundError(f'sprint3_uda not found: {sprint3_uda}')

## Cell 2 — ★ CONFIGURE THIS ★

In [ ]:
# ── Change only these three lines ────────────────────────────────────────────
MODEL_KEY   = 'nemotron-550b'   # 'llama-3-8b' | 'nemotron-550b' | 'gpt4-turbo'
DATASET     = 'tathybrid'       # 'tathybrid' | 'finhybrid' | 'nqtext' | 'fetatab' | 'papertab' | 'papertext'
PROMPT      = 'simple'          # 'simple' (zero-shot) | 'cot' (chain-of-thought)
# ─────────────────────────────────────────────────────────────────────────────

# Paths (derived automatically)
QUESTIONS_CSV = os.path.join(sprint4_root, f'benchmark/questions/{DATASET}_hard_cases.csv')
PDF_DIR       = os.path.join(sprint3_uda,  f'dataset/src_doc_files_example/{DATASET.replace("hybrid","_docs").replace("nqtext","wiki_nq_docs/pdfs").replace("fetatab","wiki_feta_docs/pdfs").replace("papertext","paper_docs").replace("papertab","paper_docs")}')
OUTPUT_DIR    = os.path.join(sprint4_root, f'experiments/{MODEL_KEY}/results')

# Verify the questions CSV exists
if not os.path.exists(QUESTIONS_CSV):
    print(f'WARNING: questions CSV not found: {QUESTIONS_CSV}')
    print('Create it first (Week 1 deliverable) before running this notebook.')
else:
    import pandas as pd
    df_check = pd.read_csv(QUESTIONS_CSV)
    print(f'Questions loaded: {len(df_check)} rows')
    print(df_check.head(3))

## Cell 3 — Run benchmark

In [ ]:
from framework.rag_runner import RAGRunner

runner = RAGRunner(model_key=MODEL_KEY, dataset=DATASET, prompt=PROMPT)

results_df = runner.run(
    questions_csv=QUESTIONS_CSV,
    pdf_dir=PDF_DIR,
    doc_col='doc_name',
    output_dir=OUTPUT_DIR,
)

print(f'\nDone. {len(results_df)} results saved.')
results_df.head()

## Cell 4 — Score results

In [ ]:
from framework.scorer import score_results
import glob

# Find the most recent results file for this combination
pattern = os.path.join(OUTPUT_DIR, f'{DATASET}_{MODEL_KEY}_{PROMPT}_*.csv')
result_files = sorted(glob.glob(pattern))

if not result_files:
    print('No results file found. Run Cell 3 first.')
else:
    latest = result_files[-1]
    print(f'Scoring: {os.path.basename(latest)}')
    info = score_results(latest, dataset=DATASET, save=True)

## Cell 5 — Quick failure analysis
Identify questions with empty or very short responses.

In [ ]:
if len(results_df) > 0:
    results_df['is_empty'] = results_df['response'].fillna('').str.strip() == ''
    results_df['resp_len'] = results_df['response'].fillna('').str.len()

    print('=== Empty responses ===')
    empty = results_df[results_df['is_empty']]
    print(f'{len(empty)} / {len(results_df)} ({len(empty)/len(results_df)*100:.1f}%)')

    if len(empty) > 0:
        print('\nSample empty questions:')
        for _, row in empty.head(5).iterrows():
            print(f'  [{row.get("question_id","?")}] {row["question"][:80]}...')

    print('\n=== Response length distribution ===')
    print(results_df['resp_len'].describe())